### Simple example of MDFC Library
1) Compress data (just samples & timestamps) from a MDF file
2) Recover samples & timestamps back into python memory

In [ ]:
# %pip install -r ../requirements.txt
# pyfastpfor will not build properly on windows using msvcc
# -> dont worry about it :) using a precompiled dll

In [ ]:
from pathlib import Path
proj_dir = Path().resolve().parent
import sys
sys.path.append(str(proj_dir))
from mdfc import (
    MDFCompressor, MDFDecompressor
)
import asammdf, numpy as np


Attempt to use windows library for testing...
Attempt to use windows library for testing decompression!...


C:\Users\staub\mdf_compression\mdfc\utils\lc_framework\get_bin.py:36: UserWarning: Cound not locate /LC-framework/lc at one or two levels above mdfc module! Therefore LC framework option has been disabled. Note that a file previously compressed with LC-framework cannot now be decompressed...
  warnings.warn(


In [7]:
# this sample data has been generated ahead of time, 
# read more about it at ./Automotive-ResearchDataSet-VIF_AEGIS/README.md
mdf_uc_path  = proj_dir / 'sample_data/Automotive-ResearchDataSet-VIF_AEGIS/mdf_uncompressed.mf4'
# size of it?
from io import BytesIO
MDF_FIL = BytesIO(
    open(mdf_uc_path, 'rb').read()
)
MDF_FIL.seek(0)
uncompressed_mdf_total_size = MDF_FIL.__sizeof__()
print(
    f'{uncompressed_mdf_total_size/1000/1000:.2f} '
    'MB Uncompressed MDF File'
)

89.10 MB Uncompressed MDF File


### Section 1: compression

In [ ]:
# how long compress it using MDF standard, and how much size?
MDF_DEFLATED = BytesIO()
with asammdf.MDF(MDF_FIL) as mdf_uc:
    mdf_uc.save(
        MDF_DEFLATED,
        overwrite=True, 
        compression=2,  # it means to transpose the data from row-oriented to col-oriented, and then compress it
    )
MDF_DEFLATED.seek(0)
deflated_mdf_total_size = MDF_DEFLATED.__sizeof__()
print(
    f'{deflated_mdf_total_size/1000/1000:.2f} '
    'MB DEFLATE-d MDF File (transpose + deflate)'
)
print(
    f'{uncompressed_mdf_total_size/deflated_mdf_total_size:.3f} '
    'CR using Deflate (MDF Standard)'
)

24.76 MB DEFLATE-d MDF File (transpose + deflate)
3.598 CR using Deflate (MDF Standard)


In [ ]:
# how about with MDFC?
# lossless compression example
MDFC_LOSSLESS = BytesIO()
MDFC_LOSSLESS.seek(0); MDF_FIL.seek(0); pass
with (
    asammdf.MDF(MDF_FIL) as mdf_fil,
    MDFCompressor(MDFC_LOSSLESS, close_file_on_exit=False) as mdfc_fil
):
    mdfc_fil.compress_all_groups(
        mdf_fil,
        on_error='raise',
    )
    # presently, must call finish function,
    #   TODO it should be done on a (successful?) __exit__
    mdfc_fil.finish()
    from copy import deepcopy
    md = deepcopy(mdfc_fil.comp_metadata)
MDFC_LOSSLESS.seek(0); MDF_FIL.seek(0); pass
mdfc_total_size = MDFC_LOSSLESS.__sizeof__()
print(
    f'{mdfc_total_size/1000/1000:.2f} '
    'MB MDFC File'
)
print(
    f'{uncompressed_mdf_total_size/mdfc_total_size:.3f} '
    'CR using MDFC (Lossless)'
)

2072 bytes of metadata
13.76 MB MDFC File
6.476 CR using Deflate (MDF Standard)


In [14]:
# is it better than deflate-2? :) ... or :(

In [ ]:
# how about with some losses?
# lossless compression example
MDFC_LOSSY = BytesIO()
MDFC_LOSSY.seek(0); MDF_FIL.seek(0); pass
time_resolution = '1ms'
with (
    asammdf.MDF(MDF_FIL) as mdf_fil,
    MDFCompressor(MDFC_LOSSY, close_file_on_exit=False) as mdfc_fil
):
    mdfc_fil.compress_all_groups(
        mdf_fil,
        on_error='raise',
        time_resolution = time_resolution,  # timestamp resolution within +/- 1ms will be discarded
                                            # in this example dataset, it effectively only removes "jitter"
                                            # if a measurement were taken faster, eg 1 sample per 10us, 
                                            # perhaps 1us or 100ns resolution could be used
    )
    # presently, must call finish function,
    #   TODO it should be done on a (successful?) __exit__
    mdfc_fil.finish()
    from copy import deepcopy
    md = deepcopy(mdfc_fil.comp_metadata)
MDFC_LOSSY.seek(0); MDF_FIL.seek(0); pass
mdfc_total_size_lossy = MDFC_LOSSY.__sizeof__()
print(
    f'{mdfc_total_size_lossy/1000/1000:.2f} '
    f'MB MDFC File (with +/- {time_resolution} loss in timestamp resolution)'
)
print(
    f'{uncompressed_mdf_total_size/mdfc_total_size_lossy:.3f} '
    f'CR using MDFC (+/- {time_resolution} Lossy)'
)

2058 bytes of metadata
8.85 MB MDFC File (with +/- 1ms loss in timestamp resolution)
10.067 CR using MDFC (+/- 1ms Lossy)


In [16]:
# is it even better? :) ... or :(

### Section 2: decompression

In [17]:
# do we recover all samples & timestamps within the specified resolution?
# only samples & timestamps are compressed in the file
# more metadata is not implemented yet

In [ ]:
# check if samples & timestamps are still close after recovery
def decompress_and_compare(sn, mdfc_fil, mdf_fil):
    # decompress the signal from mdfc
    # and compare it against the signal in mdf
    original_sig = mdf_fil.select([sn], raw=True)[0]
    original_timestamps = original_sig.timestamps
    original_samples = original_sig.samples
    
    # decompress mdfc signal
    res = mdfc_fil.decompress_signal(sn)

    # are they all close?
    assert np.allclose(
        original_timestamps, res.timestamps,
    ), f"{sn} timestamps not allclose!? :("
    assert np.allclose(
        original_samples, res.samples,
    ), f"{sn} samples not allclose!? :("


MDFC_LOSSLESS.seek(0); MDF_FIL.seek(0); pass
with (
    asammdf.MDF(MDF_FIL) as mdf_fil,
    MDFDecompressor(MDFC_LOSSLESS, close_file_on_exit=False) as mdfc_fil
):
    # mdfc_fil.decompress_time()
    try:
        # test signal decompression
        for sn in mdf_fil.channels_db.keys():
            if sn == 'time': continue  #
            print(f'on {sn}')
            decompress_and_compare(sn, mdfc_fil, mdf_fil)
            print(f'pass {sn}')
    except KeyError:
        print(f'{sn} found in MDF but not in compressed file...')
        raise  # ?
    else:
        print("All signals have passed decompression check :)")
MDFC_LOSSLESS.seek(0); MDF_FIL.seek(0); pass

on acceleration_id
pass acceleration_id
on trip_id
pass trip_id
on x_value
pass x_value
on y_value
pass y_value
on z_value
pass z_value
on gyroscope_id
pass gyroscope_id
on obdData_id
pass obdData_id
on obdPid
pass obdPid
on data
pass data
on pos_id
pass pos_id
on latitude
pass latitude
on longitude
pass longitude
on altitude
pass altitude
All signals have passed decompression check :)


### Section 3: decompression time

In [20]:
# how fast is it to read data from the MDFC file?

In [22]:
%%timeit
# testing the speed of reading the baseline MDF (without compression)
MDF_FIL.seek(0)
with (
    asammdf.MDF(MDF_FIL) as mfil,
):
    sigs = [
        sig_name # (sig_name, *chan_info) 
        for sig_name, chan_info in mfil.channels_db.items()
        if sig_name != 'time'
    ]
    for sig_sel in sigs:
        sig = mfil.select([sig_sel], raw=True)[0]

609 ms ± 112 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [24]:
%%timeit
# testing the speed of reading MDF (with deflate compression)
MDF_DEFLATED.seek(0)
with (
    asammdf.MDF(MDF_DEFLATED) as mfil,
):
    sigs = [
        sig_name # (sig_name, *chan_info) 
        for sig_name, chan_info in mfil.channels_db.items()
        if sig_name != 'time'
    ]
    for sig_sel in sigs:
        sig = mfil.select([sig_sel], raw=True)[0]

1.76 s ± 151 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [26]:
%%timeit
# testing the speed of reading the MDFC compressed file
MDFC_LOSSLESS.seek(0)
with MDFDecompressor(MDFC_LOSSLESS, close_file_on_exit=False) as dfil:
    sigs = dfil.channame_to_group.keys()
    for sn in sigs:
        res = dfil.decompress_signal(sn)

931 ms ± 99.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [27]:
%%timeit
# is there a time difference with lossy? (i dont think so...)
MDFC_LOSSY.seek(0)
with MDFDecompressor(MDFC_LOSSY, close_file_on_exit=False) as dfil:
    sigs = dfil.channame_to_group.keys()
    for sn in sigs:
        res = dfil.decompress_signal(sn)

956 ms ± 53.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
# thanks for playing!